# Soil Health Research Database Design & ETL Pipeline

**Author:** Sydney Seiter  
**Purpose:** Demonstration of database architecture for multi-site soil health research  
**Skills Demonstrated:**  
- Relational database design and normalization
- ETL (Extract, Transform, Load) pipeline development
- SQL for agricultural research data
- Data validation and quality control
- Scalable data system architecture

---

## Context

This notebook designs a production-ready database system for managing soil health research data across multiple sites. The system handles:
- Site metadata and characteristics
- Field sampling records
- Laboratory analysis results
- Crop management practices
- Weather and environmental data
- Quality control flags and audit trails

This demonstrates the database management and data pipeline skills essential for coordinating the Soil Health Institute's research network.

## Setup: Configure Output Directory

Choose where to save output files. Uncomment the Google Drive option if you want to save to Drive.

In [ ]:
import os

# OPTION 1: Save to current directory (default)
output_dir = '.'

# OPTION 2: Create a dedicated output folder
# output_dir = 'soil_health_outputs'
# os.makedirs(output_dir, exist_ok=True)

# OPTION 3: Save to Google Drive (uncomment to use)
# from google.colab import drive
# drive.mount('/content/drive')
# output_dir = '/content/drive/MyDrive/SoilHealthPortfolio'
# os.makedirs(output_dir, exist_ok=True)

print(f"✓ Outputs will be saved to: {output_dir}")

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully")
print(f"Analysis date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 1. Database Schema Design

Designing normalized relational database following best practices for research data management.

In [ ]:
# Connect to SQLite database (production would use PostgreSQL)
db_path = os.path.join(output_dir, 'soil_health_research.db')
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

# Drop existing tables if they exist (for demo purposes)
cursor.execute("DROP TABLE IF EXISTS weather_data")
cursor.execute("DROP TABLE IF EXISTS management_practices")
cursor.execute("DROP TABLE IF EXISTS lab_results")
cursor.execute("DROP TABLE IF EXISTS field_samples")
cursor.execute("DROP TABLE IF EXISTS plots")
cursor.execute("DROP TABLE IF EXISTS research_sites")
cursor.execute("DROP TABLE IF EXISTS soil_taxonomy")

print("Creating database schema...\n")

# Table 1: Research Sites (site-level metadata)
cursor.execute('''
CREATE TABLE research_sites (
    site_id TEXT PRIMARY KEY,
    site_name TEXT NOT NULL,
    state TEXT NOT NULL,
    latitude REAL,
    longitude REAL,
    elevation_m REAL,
    climate_zone TEXT,
    mlra TEXT,  -- Major Land Resource Area
    established_date DATE,
    principal_investigator TEXT,
    institution TEXT,
    contact_email TEXT,
    notes TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    updated_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
)
''')
print("✓ Created table: research_sites")

# Table 2: Soil Taxonomy (characterization data)
cursor.execute('''
CREATE TABLE soil_taxonomy (
    taxonomy_id INTEGER PRIMARY KEY AUTOINCREMENT,
    site_id TEXT NOT NULL,
    plot_id TEXT,  -- Can be site-level or plot-specific
    soil_order TEXT,
    soil_suborder TEXT,
    soil_great_group TEXT,
    soil_series TEXT,
    texture_class TEXT,
    sand_pct REAL,
    silt_pct REAL,
    clay_pct REAL,
    drainage_class TEXT,
    parent_material TEXT,
    FOREIGN KEY (site_id) REFERENCES research_sites(site_id)
)
''')
print("✓ Created table: soil_taxonomy")

# Table 3: Plots (experimental units)
cursor.execute('''
CREATE TABLE plots (
    plot_id TEXT PRIMARY KEY,
    site_id TEXT NOT NULL,
    block INTEGER,
    treatment TEXT,
    plot_area_m2 REAL,
    slope_pct REAL,
    aspect TEXT,
    previous_crop TEXT,
    notes TEXT,
    FOREIGN KEY (site_id) REFERENCES research_sites(site_id)
)
''')
print("✓ Created table: plots")

# Table 4: Field Samples (sampling events)
cursor.execute('''
CREATE TABLE field_samples (
    sample_id TEXT PRIMARY KEY,
    plot_id TEXT NOT NULL,
    collection_date DATE NOT NULL,
    collector_name TEXT,
    sample_type TEXT,  -- soil, plant tissue, water, etc.
    depth_top_cm REAL,
    depth_bottom_cm REAL,
    latitude REAL,
    longitude REAL,
    field_moisture_pct REAL,
    field_notes TEXT,
    chain_of_custody TEXT,
    qa_flag TEXT DEFAULT 'PENDING',
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
    FOREIGN KEY (plot_id) REFERENCES plots(plot_id)
)
''')
print("✓ Created table: field_samples")

# Table 5: Lab Results (analysis data)
cursor.execute('''
CREATE TABLE lab_results (
    result_id INTEGER PRIMARY KEY AUTOINCREMENT,
    sample_id TEXT NOT NULL,
    lab_name TEXT NOT NULL,
    analysis_date DATE,
    analyst_name TEXT,
    -- Soil chemistry
    ph_water REAL,
    ph_method TEXT,
    organic_matter_pct REAL,
    om_method TEXT,
    total_carbon_pct REAL,
    total_nitrogen_pct REAL,
    c_n_ratio REAL,
    -- Extractable nutrients
    phosphorus_ppm REAL,
    p_method TEXT,  -- Critical: Mehlich-3, Bray, Olsen, etc.
    potassium_ppm REAL,
    calcium_ppm REAL,
    magnesium_ppm REAL,
    sulfur_ppm REAL,
    -- Cation exchange
    cec_meq_100g REAL,
    -- Soil biology
    microbial_biomass_c_mg_kg REAL,
    microbial_biomass_n_mg_kg REAL,
    soil_respiration_mg_co2_kg_day REAL,
    -- Physical properties
    bulk_density_g_cm3 REAL,
    aggregate_stability_pct REAL,
    infiltration_cm_hr REAL,
    water_holding_capacity_pct REAL,
    -- QC fields
    quality_control_passed BOOLEAN,
    qc_notes TEXT,
    instrument_id TEXT,
    FOREIGN KEY (sample_id) REFERENCES field_samples(sample_id)
)
''')
print("✓ Created table: lab_results")

# Table 6: Management Practices (agronomic activities)
cursor.execute('''
CREATE TABLE management_practices (
    practice_id INTEGER PRIMARY KEY AUTOINCREMENT,
    plot_id TEXT NOT NULL,
    practice_date DATE NOT NULL,
    practice_type TEXT NOT NULL,  -- tillage, planting, harvest, fertilizer, etc.
    crop_type TEXT,
    variety TEXT,
    tillage_type TEXT,
    tillage_depth_cm REAL,
    fertilizer_type TEXT,
    fertilizer_rate_kg_ha REAL,
    n_rate_kg_ha REAL,
    p_rate_kg_ha REAL,
    k_rate_kg_ha REAL,
    cover_crop_species TEXT,
    seeding_rate_kg_ha REAL,
    harvest_yield_kg_ha REAL,
    notes TEXT,
    FOREIGN KEY (plot_id) REFERENCES plots(plot_id)
)
''')
print("✓ Created table: management_practices")

# Table 7: Weather Data (environmental monitoring)
cursor.execute('''
CREATE TABLE weather_data (
    weather_id INTEGER PRIMARY KEY AUTOINCREMENT,
    site_id TEXT NOT NULL,
    observation_date DATE NOT NULL,
    temp_max_c REAL,
    temp_min_c REAL,
    temp_avg_c REAL,
    precipitation_mm REAL,
    solar_radiation_mj_m2 REAL,
    wind_speed_m_s REAL,
    relative_humidity_pct REAL,
    data_source TEXT,  -- station, NLDAS, DAYMET, etc.
    FOREIGN KEY (site_id) REFERENCES research_sites(site_id)
)
''')
print("✓ Created table: weather_data")

# Create indexes for common queries
cursor.execute('CREATE INDEX idx_samples_plot ON field_samples(plot_id)')
cursor.execute('CREATE INDEX idx_samples_date ON field_samples(collection_date)')
cursor.execute('CREATE INDEX idx_results_sample ON lab_results(sample_id)')
cursor.execute('CREATE INDEX idx_weather_site_date ON weather_data(site_id, observation_date)')
cursor.execute('CREATE INDEX idx_practices_plot_date ON management_practices(plot_id, practice_date)')

conn.commit()

print("\n✅ Database schema created successfully")
print("\nSchema includes:")
print("  - 7 core tables with proper relationships")
print("  - Foreign key constraints for referential integrity")
print("  - Indexes for query optimization")
print("  - Audit trail fields (created_at, updated_at)")
print("  - QA/QC flag fields for data quality tracking")

## 2. Generate Sample Research Data

Creating realistic multi-site research data to demonstrate ETL pipeline.

In [ ]:
# Generate sample data for 3 research sites
np.random.seed(42)

# Sites data
sites_data = pd.DataFrame([
    {
        'site_id': 'NC-001',
        'site_name': 'Tidewater Research Station',
        'state': 'North Carolina',
        'latitude': 35.9132,
        'longitude': -77.0420,
        'elevation_m': 12,
        'climate_zone': 'Humid Subtropical',
        'mlra': '153A',
        'established_date': '2022-05-15',
        'principal_investigator': 'Dr. Sarah Johnson',
        'institution': 'NC State University',
        'contact_email': 'sjohnson@ncsu.edu',
        'notes': 'Sandy Coastal Plain soil, no-till corn-soybean rotation study'
    },
    {
        'site_id': 'IA-001',
        'site_name': 'Central Iowa Research Farm',
        'state': 'Iowa',
        'latitude': 42.0308,
        'longitude': -93.6319,
        'elevation_m': 305,
        'climate_zone': 'Humid Continental',
        'mlra': '108C',
        'established_date': '2022-05-01',
        'principal_investigator': 'Dr. Michael Chen',
        'institution': 'Iowa State University',
        'contact_email': 'mchen@iastate.edu',
        'notes': 'Deep prairie mollisols, corn-soybean-oats rotation'
    },
    {
        'site_id': 'MT-001',
        'site_name': 'Northern Plains Research Site',
        'state': 'Montana',
        'latitude': 48.5501,
        'longitude': -109.6841,
        'elevation_m': 914,
        'climate_zone': 'Semi-Arid Continental',
        'mlra': '52',
        'established_date': '2022-06-01',
        'principal_investigator': 'Dr. Emily Rodriguez',
        'institution': 'Montana State University',
        'contact_email': 'erodriguez@montana.edu',
        'notes': 'Semi-arid grassland, spring wheat-fallow system'
    }
])

# Soil taxonomy data
taxonomy_data = pd.DataFrame([
    {
        'site_id': 'NC-001',
        'plot_id': None,  # Site-level characterization
        'soil_order': 'Ultisols',
        'soil_suborder': 'Udults',
        'soil_great_group': 'Hapludults',
        'soil_series': 'Norfolk',
        'texture_class': 'Sandy loam',
        'sand_pct': 65.0,
        'silt_pct': 25.0,
        'clay_pct': 10.0,
        'drainage_class': 'Well drained',
        'parent_material': 'Marine sediments'
    },
    {
        'site_id': 'IA-001',
        'plot_id': None,
        'soil_order': 'Mollisols',
        'soil_suborder': 'Udolls',
        'soil_great_group': 'Hapludolls',
        'soil_series': 'Clarion',
        'texture_class': 'Loam',
        'sand_pct': 40.0,
        'silt_pct': 40.0,
        'clay_pct': 20.0,
        'drainage_class': 'Well drained',
        'parent_material': 'Glacial till'
    },
    {
        'site_id': 'MT-001',
        'plot_id': None,
        'soil_order': 'Mollisols',
        'soil_suborder': 'Ustolls',
        'soil_great_group': 'Haplustolls',
        'soil_series': 'Williams',
        'texture_class': 'Clay loam',
        'sand_pct': 30.0,
        'silt_pct': 35.0,
        'clay_pct': 35.0,
        'drainage_class': 'Well drained',
        'parent_material': 'Glacial till'
    }
])

# Generate plots (8 per site: 4 blocks × 2 treatments)
plots_list = []
for site in sites_data['site_id']:
    for block in [1, 2, 3, 4]:
        for treatment in ['No-Till', 'Conventional']:
            plots_list.append({
                'plot_id': f"{site}-B{block}-{treatment[:4]}",
                'site_id': site,
                'block': block,
                'treatment': treatment,
                'plot_area_m2': 100.0,
                'slope_pct': np.random.uniform(0.5, 3.0),
                'aspect': np.random.choice(['N', 'S', 'E', 'W', 'NE', 'NW', 'SE', 'SW']),
                'previous_crop': np.random.choice(['Corn', 'Soybean', 'Wheat']),
                'notes': f'Block {block}, {treatment} treatment'
            })

plots_data = pd.DataFrame(plots_list)

print(f"Generated data:")
print(f"  - {len(sites_data)} research sites")
print(f"  - {len(taxonomy_data)} soil characterizations")
print(f"  - {len(plots_data)} experimental plots")
print(f"\nPlots per site: {len(plots_data) // len(sites_data)}")
print(f"Blocks: {plots_data['block'].nunique()}")
print(f"Treatments: {plots_data['treatment'].nunique()}")

## 3. ETL Pipeline: Extract from Source Files

Simulating data extraction from lab reports and field collection sheets.

In [ ]:
# Simulate field sampling data (as might come from field sheets)
samples_list = []
sample_counter = 1

for plot_id in plots_data['plot_id']:
    site_id = plot_id.split('-')[0] + '-001'
    
    # 3 sampling dates per plot (spring 2023, 2024, 2025)
    for year in [2023, 2024, 2025]:
        sample_date = f"{year}-04-{15 + np.random.randint(0, 10):02d}"
        
        samples_list.append({
            'sample_id': f"S{year}-{sample_counter:04d}",
            'plot_id': plot_id,
            'collection_date': sample_date,
            'collector_name': np.random.choice(['J. Smith', 'M. Garcia', 'A. Patel']),
            'sample_type': 'soil',
            'depth_top_cm': 0,
            'depth_bottom_cm': 15,
            'latitude': sites_data[sites_data['site_id']==site_id]['latitude'].values[0] + 
                       np.random.uniform(-0.001, 0.001),
            'longitude': sites_data[sites_data['site_id']==site_id]['longitude'].values[0] + 
                        np.random.uniform(-0.001, 0.001),
            'field_moisture_pct': np.random.uniform(15, 25),
            'field_notes': f'Standard soil health sampling, {year} spring',
            'chain_of_custody': 'Field → Cooler → Lab within 24hr',
            'qa_flag': 'PENDING'
        })
        sample_counter += 1

samples_data = pd.DataFrame(samples_list)

print(f"Extracted {len(samples_data)} field samples")
print(f"Date range: {samples_data['collection_date'].min()} to {samples_data['collection_date'].max()}")
print(f"Samples per plot: {len(samples_data) // len(plots_data)}")
print("\nSample of field data:")
print(samples_data.head())

## 4. ETL Pipeline: Transform and Validate

Apply data validation rules and transformations before loading.

In [ ]:
def validate_and_transform_samples(samples_df):
    """
    Apply business rules and validation to sample data.
    """
    df = samples_df.copy()
    validation_log = []
    
    # Rule 1: Collection date must be valid
    df['collection_date'] = pd.to_datetime(df['collection_date'], errors='coerce')
    invalid_dates = df['collection_date'].isna().sum()
    if invalid_dates > 0:
        validation_log.append(f"WARNING: {invalid_dates} invalid dates detected")
    
    # Rule 2: Depth must be logical (top < bottom)
    invalid_depth = (df['depth_top_cm'] >= df['depth_bottom_cm']).sum()
    if invalid_depth > 0:
        validation_log.append(f"WARNING: {invalid_depth} samples with invalid depth range")
        df.loc[df['depth_top_cm'] >= df['depth_bottom_cm'], 'qa_flag'] = 'FAIL_depth'
    
    # Rule 3: Geographic coordinates must be reasonable
    invalid_lat = ((df['latitude'] < 25) | (df['latitude'] > 50)).sum()
    invalid_lon = ((df['longitude'] < -130) | (df['longitude'] > -60)).sum()
    if invalid_lat > 0 or invalid_lon > 0:
        validation_log.append(f"WARNING: {invalid_lat + invalid_lon} samples with invalid coordinates")
        df.loc[(df['latitude'] < 25) | (df['latitude'] > 50) | 
               (df['longitude'] < -130) | (df['longitude'] > -60), 'qa_flag'] = 'FAIL_coordinates'
    
    # Rule 4: Sample IDs must be unique
    duplicates = df['sample_id'].duplicated().sum()
    if duplicates > 0:
        validation_log.append(f"ERROR: {duplicates} duplicate sample IDs detected")
    
    # Rule 5: Plot IDs must exist in plots table
    valid_plots = plots_data['plot_id'].unique()
    invalid_plots = (~df['plot_id'].isin(valid_plots)).sum()
    if invalid_plots > 0:
        validation_log.append(f"ERROR: {invalid_plots} samples reference non-existent plots")
    
    # Update QA flag for passed samples
    df.loc[df['qa_flag'] == 'PENDING', 'qa_flag'] = 'PASS'
    
    return df, validation_log

# Run validation
samples_validated, validation_log = validate_and_transform_samples(samples_data)

print("Data Validation Results:")
print("=" * 60)
if validation_log:
    for log in validation_log:
        print(f"  {log}")
else:
    print("  ✓ All validation checks passed")

print(f"\nQA Flag Summary:")
print(samples_validated['qa_flag'].value_counts())
print(f"\nPass rate: {(samples_validated['qa_flag']=='PASS').sum() / len(samples_validated) * 100:.1f}%")

## 5. ETL Pipeline: Load to Database

Insert validated data into database with proper error handling.

In [ ]:
# Load data into database tables
print("Loading data into database...\n")

try:
    # Load sites
    sites_data.to_sql('research_sites', conn, if_exists='append', index=False)
    print(f"✓ Loaded {len(sites_data)} research sites")
    
    # Load soil taxonomy
    taxonomy_data.to_sql('soil_taxonomy', conn, if_exists='append', index=False)
    print(f"✓ Loaded {len(taxonomy_data)} soil characterizations")
    
    # Load plots
    plots_data.to_sql('plots', conn, if_exists='append', index=False)
    print(f"✓ Loaded {len(plots_data)} experimental plots")
    
    # Load samples (only those that passed QA)
    samples_to_load = samples_validated[samples_validated['qa_flag'] == 'PASS'].copy()
    samples_to_load.to_sql('field_samples', conn, if_exists='append', index=False)
    print(f"✓ Loaded {len(samples_to_load)} field samples (QA passed)")
    
    # Log failed samples
    failed_samples = samples_validated[samples_validated['qa_flag'] != 'PASS']
    if len(failed_samples) > 0:
        print(f"⚠ {len(failed_samples)} samples failed QA - logged for review")
        failed_samples.to_csv(os.path.join(output_dir, 'failed_samples_log.csv'), index=False)
    
    conn.commit()
    print("\n✅ ETL pipeline completed successfully")
    
except Exception as e:
    conn.rollback()
    print(f"❌ Error during data load: {str(e)}")
    raise

## 6. Generate Lab Results Data

Simulate laboratory analysis results with realistic soil chemistry values.

In [ ]:
# Generate realistic lab results for each sample
lab_results_list = []

for idx, sample in samples_to_load.iterrows():
    # Determine site characteristics
    site_id = sample['plot_id'].split('-')[0] + '-001'
    treatment = 'No-Till' if 'No-T' in sample['plot_id'] else 'Conventional'
    year = int(sample['sample_id'][1:5])
    
    # Site-specific baselines
    if 'NC' in site_id:
        om_base = 2.0
        ph_base = 5.8
        p_base = 25
    elif 'IA' in site_id:
        om_base = 4.0
        ph_base = 6.5
        p_base = 40
    else:  # MT
        om_base = 3.0
        ph_base = 7.8
        p_base = 18
    
    # Treatment and year effects
    year_effect = (year - 2023) * 0.1
    treatment_effect = 1.15 if treatment == 'No-Till' else 1.0
    
    # Generate values with natural variation
    lab_results_list.append({
        'sample_id': sample['sample_id'],
        'lab_name': 'Soil Health Institute Analytical Lab',
        'analysis_date': (pd.to_datetime(sample['collection_date']) + timedelta(days=14)).strftime('%Y-%m-%d'),
        'analyst_name': np.random.choice(['Lab Tech 1', 'Lab Tech 2', 'Lab Tech 3']),
        'ph_water': np.clip(ph_base + np.random.normal(0, 0.3), 4.0, 9.0),
        'ph_method': '1:1 soil:water',
        'organic_matter_pct': np.clip((om_base * treatment_effect + year_effect) + 
                                     np.random.normal(0, 0.2), 0.5, 10),
        'om_method': 'Loss on ignition',
        'total_carbon_pct': None,
        'total_nitrogen_pct': None,
        'c_n_ratio': None,
        'phosphorus_ppm': np.clip(p_base * treatment_effect + np.random.normal(0, 8), 5, 200),
        'p_method': 'Mehlich-3' if 'NC' in site_id else 'Bray P-1' if 'IA' in site_id else 'Olsen',
        'potassium_ppm': np.clip(150 * treatment_effect + np.random.normal(0, 30), 50, 500),
        'calcium_ppm': np.clip(800 + np.random.normal(0, 150), 200, 5000),
        'magnesium_ppm': np.clip(180 + np.random.normal(0, 40), 50, 800),
        'sulfur_ppm': np.clip(12 + np.random.normal(0, 3), 3, 50),
        'cec_meq_100g': np.clip((8 if 'NC' in site_id else 18 if 'IA' in site_id else 12) + 
                               np.random.normal(0, 2), 3, 40),
        'microbial_biomass_c_mg_kg': np.clip(300 * treatment_effect + year_effect * 20 + 
                                            np.random.normal(0, 50), 100, 800),
        'microbial_biomass_n_mg_kg': None,
        'soil_respiration_mg_co2_kg_day': None,
        'bulk_density_g_cm3': np.clip(1.45 / treatment_effect + np.random.normal(0, 0.05), 1.0, 1.8),
        'aggregate_stability_pct': np.clip(45 * treatment_effect + year_effect * 3 + 
                                          np.random.normal(0, 5), 15, 90),
        'infiltration_cm_hr': np.clip(1.5 * treatment_effect + year_effect * 0.2 + 
                                     np.random.normal(0, 0.3), 0.5, 5),
        'water_holding_capacity_pct': None,
        'quality_control_passed': True,
        'qc_notes': 'Standard QC protocols passed',
        'instrument_id': f'INST-{np.random.randint(1, 5):02d}'
    })

lab_results_data = pd.DataFrame(lab_results_list)

# Load lab results to database
lab_results_data.to_sql('lab_results', conn, if_exists='append', index=False)
conn.commit()

print(f"✓ Generated and loaded {len(lab_results_data)} laboratory analysis results")
print(f"\nLab turnaround time: ~14 days from sample collection")
print(f"Analysis methods documented: pH, OM, P extraction")
print(f"QC pass rate: {lab_results_data['quality_control_passed'].sum() / len(lab_results_data) * 100:.1f}%")

## 7. SQL Queries for Research Support

Demonstrating complex queries typical for multi-site research.

In [ ]:
# Query 1: Treatment comparison across sites
query1 = """
SELECT 
    s.site_name,
    s.state,
    p.treatment,
    COUNT(DISTINCT fs.sample_id) as sample_count,
    ROUND(AVG(lr.organic_matter_pct), 2) as avg_om_pct,
    ROUND(AVG(lr.aggregate_stability_pct), 1) as avg_agg_stability,
    ROUND(AVG(lr.bulk_density_g_cm3), 3) as avg_bulk_density
FROM research_sites s
JOIN plots p ON s.site_id = p.site_id
JOIN field_samples fs ON p.plot_id = fs.plot_id
JOIN lab_results lr ON fs.sample_id = lr.sample_id
WHERE fs.qa_flag = 'PASS'
GROUP BY s.site_name, s.state, p.treatment
ORDER BY s.state, p.treatment
"""

results1 = pd.read_sql_query(query1, conn)
print("Query 1: Treatment Effects by Site")
print("=" * 80)
print(results1.to_string(index=False))

# Query 2: Temporal trends
query2 = """
SELECT 
    strftime('%Y', fs.collection_date) as year,
    p.treatment,
    COUNT(*) as samples,
    ROUND(AVG(lr.organic_matter_pct), 2) as avg_om,
    ROUND(AVG(lr.microbial_biomass_c_mg_kg), 1) as avg_mbc
FROM field_samples fs
JOIN plots p ON fs.plot_id = p.plot_id
JOIN lab_results lr ON fs.sample_id = lr.sample_id
GROUP BY year, p.treatment
ORDER BY year, p.treatment
"""

results2 = pd.read_sql_query(query2, conn)
print("\n\nQuery 2: Temporal Trends (2023-2025)")
print("=" * 80)
print(results2.to_string(index=False))

# Query 3: Data completeness check
query3 = """
SELECT 
    s.site_name,
    COUNT(DISTINCT p.plot_id) as total_plots,
    COUNT(DISTINCT fs.sample_id) as total_samples,
    COUNT(DISTINCT lr.result_id) as total_results,
    ROUND(COUNT(DISTINCT lr.result_id) * 1.0 / COUNT(DISTINCT fs.sample_id), 2) as lab_completion_rate
FROM research_sites s
LEFT JOIN plots p ON s.site_id = p.site_id
LEFT JOIN field_samples fs ON p.plot_id = fs.plot_id
LEFT JOIN lab_results lr ON fs.sample_id = lr.sample_id
GROUP BY s.site_name
"""

results3 = pd.read_sql_query(query3, conn)
print("\n\nQuery 3: Data Completeness by Site")
print("=" * 80)
print(results3.to_string(index=False))

# Query 4: Soil taxonomy summary
query4 = """
SELECT 
    s.state,
    st.soil_order,
    st.soil_series,
    st.texture_class,
    st.drainage_class,
    ROUND(AVG(lr.organic_matter_pct), 2) as avg_baseline_om
FROM research_sites s
JOIN soil_taxonomy st ON s.site_id = st.site_id
LEFT JOIN plots p ON s.site_id = p.site_id
LEFT JOIN field_samples fs ON p.plot_id = fs.plot_id
LEFT JOIN lab_results lr ON fs.sample_id = lr.sample_id
GROUP BY s.state, st.soil_order, st.soil_series
"""

results4 = pd.read_sql_query(query4, conn)
print("\n\nQuery 4: Soil Characterization and Baseline OM")
print("=" * 80)
print(results4.to_string(index=False))

## 8. Database Quality Metrics Dashboard

Visualizing data completeness and quality across the research network.

In [ ]:
# Create database quality dashboard
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Research Database Quality Metrics Dashboard', fontsize=16, fontweight='bold')

# Plot 1: Samples by site and year
samples_by_site_year = pd.read_sql_query("""
    SELECT 
        s.state,
        strftime('%Y', fs.collection_date) as year,
        COUNT(*) as sample_count
    FROM field_samples fs
    JOIN plots p ON fs.plot_id = p.plot_id
    JOIN research_sites s ON p.site_id = s.site_id
    GROUP BY s.state, year
""", conn)

samples_pivot = samples_by_site_year.pivot(index='year', columns='state', values='sample_count')
samples_pivot.plot(kind='bar', ax=axes[0,0], width=0.8)
axes[0,0].set_title('Sample Collection by Site and Year', fontweight='bold')
axes[0,0].set_xlabel('Year')
axes[0,0].set_ylabel('Number of Samples')
axes[0,0].legend(title='State')
axes[0,0].grid(axis='y', alpha=0.3)

# Plot 2: Data completeness
completeness_data = pd.read_sql_query("""
    SELECT 
        s.state,
        COUNT(DISTINCT fs.sample_id) as samples_collected,
        COUNT(DISTINCT lr.result_id) as lab_results_received
    FROM research_sites s
    JOIN plots p ON s.site_id = p.site_id
    LEFT JOIN field_samples fs ON p.plot_id = fs.plot_id
    LEFT JOIN lab_results lr ON fs.sample_id = lr.sample_id
    GROUP BY s.state
""", conn)

x = np.arange(len(completeness_data))
width = 0.35
axes[0,1].bar(x - width/2, completeness_data['samples_collected'], width, label='Samples Collected')
axes[0,1].bar(x + width/2, completeness_data['lab_results_received'], width, label='Lab Results')
axes[0,1].set_title('Data Completeness by Site', fontweight='bold')
axes[0,1].set_ylabel('Count')
axes[0,1].set_xticks(x)
axes[0,1].set_xticklabels(completeness_data['state'])
axes[0,1].legend()
axes[0,1].grid(axis='y', alpha=0.3)

# Plot 3: Treatment distribution
treatment_dist = pd.read_sql_query("""
    SELECT 
        s.state,
        p.treatment,
        COUNT(DISTINCT fs.sample_id) as sample_count
    FROM research_sites s
    JOIN plots p ON s.site_id = p.site_id
    JOIN field_samples fs ON p.plot_id = fs.plot_id
    GROUP BY s.state, p.treatment
""", conn)

treatment_pivot = treatment_dist.pivot(index='state', columns='treatment', values='sample_count')
treatment_pivot.plot(kind='bar', ax=axes[1,0], stacked=True, width=0.7)
axes[1,0].set_title('Samples by Treatment and Site', fontweight='bold')
axes[1,0].set_xlabel('State')
axes[1,0].set_ylabel('Number of Samples')
axes[1,0].legend(title='Treatment')
axes[1,0].grid(axis='y', alpha=0.3)
plt.setp(axes[1,0].xaxis.get_majorticklabels(), rotation=0)

# Plot 4: QA flag distribution
qa_summary = pd.read_sql_query("""
    SELECT qa_flag, COUNT(*) as count
    FROM field_samples
    GROUP BY qa_flag
""", conn)

axes[1,1].pie(qa_summary['count'], labels=qa_summary['qa_flag'], autopct='%1.1f%%',
             colors=['#2ecc71', '#e74c3c', '#f39c12'])
axes[1,1].set_title('Sample QA Status Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(output_dir, 'database_quality_dashboard.png'), dpi=150, bbox_inches='tight')
plt.show()

print("Database quality dashboard generated")

## 9. Database Export and Documentation

Creating data dictionary and export utilities.

In [ ]:
# Generate comprehensive data dictionary
table_schemas = []

# Get all table names
tables = pd.read_sql_query("""
    SELECT name FROM sqlite_master WHERE type='table' ORDER BY name
""", conn)

for table in tables['name']:
    # Get column information
    schema = pd.read_sql_query(f"PRAGMA table_info({table})", conn)
    schema['table_name'] = table
    table_schemas.append(schema[['table_name', 'name', 'type', 'notnull', 'pk']])

full_schema = pd.concat(table_schemas, ignore_index=True)
full_schema.columns = ['Table', 'Column', 'Data Type', 'Not Null', 'Primary Key']

# Save data dictionary
full_schema.to_csv(os.path.join(output_dir, 'database_data_dictionary.csv'), index=False)

print("Database Documentation Generated:")
print("=" * 80)
print(f"Total tables: {tables['name'].nunique()}")
print(f"Total columns: {len(full_schema)}")
print("\nTable Summary:")
for table in tables['name']:
    col_count = len(full_schema[full_schema['Table'] == table])
    row_count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table}", conn)['cnt'].values[0]
    print(f"  {table:30s}: {col_count:2d} columns, {row_count:4d} rows")

print("\n✅ Data dictionary saved to: database_data_dictionary.csv")

# Export sample dataset for analysis
export_query = """
SELECT 
    s.site_name,
    s.state,
    p.plot_id,
    p.treatment,
    p.block,
    fs.sample_id,
    fs.collection_date,
    lr.organic_matter_pct,
    lr.ph_water,
    lr.phosphorus_ppm,
    lr.aggregate_stability_pct,
    lr.bulk_density_g_cm3,
    lr.microbial_biomass_c_mg_kg
FROM research_sites s
JOIN plots p ON s.site_id = p.site_id
JOIN field_samples fs ON p.plot_id = fs.plot_id
JOIN lab_results lr ON fs.sample_id = lr.sample_id
WHERE fs.qa_flag = 'PASS'
ORDER BY s.state, p.plot_id, fs.collection_date
"""

export_data = pd.read_sql_query(export_query, conn)
export_data.to_csv(os.path.join(output_dir, 'soil_health_research_export.csv'), index=False)

print(f"\n✅ Research dataset exported: {len(export_data)} observations")
print("   File: soil_health_research_export.csv")

## Summary

This notebook demonstrates comprehensive database management skills:

✅ **Relational database design** - Properly normalized schema with referential integrity  
✅ **ETL pipeline development** - Extract, validate, transform, and load workflows  
✅ **Data validation** - Business rules and QA/QC protocols  
✅ **SQL expertise** - Complex queries for multi-site research  
✅ **Data quality tracking** - Audit trails and QA flag systems  
✅ **Scalable architecture** - Designed for growing research network  
✅ **Documentation** - Data dictionaries and metadata management  
✅ **Agricultural domain knowledge** - Appropriate data structures for soil research  

**Direct application to Soil Health Institute:**
- Database design for multi-site soil health monitoring
- ETL pipelines for laboratory and field data integration
- Quality control systems for research data
- SQL queries supporting research analysis and reporting
- Data export utilities for collaborators and publications
- Scalable architecture supporting network expansion

**Production deployment recommendations:**
- Migrate to PostgreSQL for production (demonstrated with SQLite for portability)
- Implement role-based access control for multi-institution collaboration
- Add automated backup and disaster recovery procedures
- Develop web-based data entry interface for field technicians
- Integrate with LIMS (Laboratory Information Management System)
- Add API layer for external tool integration

In [ ]:
# Close database connection
conn.close()
print("\n✅ Database connection closed")
print("\nNotebook execution complete.")
print(f"\nAll outputs saved to: {output_dir}")